# Bài 32 — chuỗi luyện Easy · LeetCode #168 — Excel Sheet Column Title

**Giáo trình Jupyter Notebook tiếng Việt cho người mới bắt đầu · Python 3**  
**Link đề:** https://leetcode.com/problems/excel-sheet-column-title/ · **Mức độ:** Easy · **Chủ điểm:** Math · bijective base-26 · chia lấy dư

**Kết quả bạn đạt được sau bài học**
1. Phát biểu được chính xác input, output, và những điều kiện đề bảo đảm; phân biệt dữ liệu thật với ví dụ mô phỏng.
2. Tự diễn giải công cụ Python liên quan bằng chương trình mini; không ghi nhớ cú pháp một cách máy móc.
3. Tự nghĩ ra một cách giải dễ hiểu, thấy rõ điểm hạn chế, rồi suy ra cách chính tốt hơn.
4. Lập bảng và chạy chương trình `trace_*` để thấy trạng thái thay đổi từng bước, kể cả nhánh đặc biệt.
5. Giải thích được tính đúng bằng **bất biến (invariant)**, tự suy luận Time/Space Complexity.
6. Viết lại `class Solution` để nộp LeetCode và tự giải 6 bài tập, có gợi ý ba mức và đáp án cuối.

> **Cách học:** Đọc một markdown → tự dự đoán kết quả code → chạy cell → so sánh → tự giải thích vì sao. Các cell TODO chỉ định nghĩa hàm, không được chạy tự động.


## 1. Đề bài, output và ví dụ chính xác

Cho số nguyên dương `columnNumber`, trả về **chuỗi tiêu đề cột Excel**: `A = 1`, `B = 2`, ..., `Z = 26`, `AA = 27`, `AB = 28`, ..., `ZY = 701`. Đề bảo đảm `1 <= columnNumber <= 2^31 - 1`; vì vậy ta **không cần** đặt quy tắc cho `n=0` hay số âm.

Các ví dụ LeetCode: `1 → "A"`, `28 → "AB"`, `701 → "ZY"`. Một ca quan trọng: `26 → "Z"`, còn `27 → "AA"`; đây là điều kiện biên khiến chia thông thường theo cơ số 26 dễ sai. Kết quả là **string**, không phải số.

Excel khác hệ đếm vị trí thông thường: không có chữ số 0 đại diện cho phần dư 0, chỉ có ký hiệu 1..26. Ta sẽ giải thích điều này bằng các thí nghiệm `divmod` nhỏ, sau đó mới rút ra công thức `n -= 1` trước khi chia.


## 2. Thiết lập môi trường để chạy notebook

Notebook dùng **Python 3** chuẩn, không cần cài thêm package. Hãy chạy **từ trên xuống dưới** để các tên hàm đã được định nghĩa trước khi gọi. Không copy `print(...)` vào LeetCode; các dòng in ở đây chỉ phục vụ việc học.

**Phân biệt ba lớp:** (1) demo nhỏ để hiểu công cụ, (2) hàm minh họa giải bài trong notebook, (3) `class Solution` đúng chữ ký để nộp trên trang đề. Với bài #168, ta sẽ học kỹ thuật **Math · bijective base-26 · chia lấy dư** theo đúng thứ tự này.

Từng code cell sau đây cần cho ra output ngắn và có thể kiểm tra bằng mắt. Trong cell kiểm thử, `assert` không có output nghĩa là điều kiện đã đúng; nếu sai, Python báo `AssertionError`.


In [1]:
from __future__ import annotations
import random
ALPHABET='ABCDEFGHIJKLMNOPQRSTUVWXYZ'
print('Python 3; bảng mã:',ALPHABET)
print('A=',1,'Z=',26,'AA=',27)


Python 3; bảng mã: ABCDEFGHIJKLMNOPQRSTUVWXYZ
A= 1 Z= 26 AA= 27


### 2.1. Vị trí ký tự: `A` có chỉ số 0 nhưng là cột 1

Chuỗi trong Python đánh chỉ số từ **0**, trong Excel cột A được đánh số **1**. Hai hệ quy chiếu khác nhau. Trong `ALPHABET`, `ALPHABET[0]` là A, `ALPHABET[25]` là Z. Luôn xác định rõ khi dùng `0..25` và khi dùng `1..26`.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [2]:
print('index 0:',ALPHABET[0])
print('index 25:',ALPHABET[25])
for i in [1,2,26]:print('cột',i,'=',ALPHABET[i-1])


index 0: A
index 25: Z
cột 1 = A
cột 2 = B
cột 26 = Z


### 2.2. `%` và `//` là hai phần của phép chia

`n // 26` cho thương nguyên, `n % 26` cho phần dư; Python bảo đảm `n == (n//26)*26 + n%26` với n không âm. Hãy xem 28 // 26 = 1 và 28 % 26 = 2; nhưng bài này phải dịch về 0-based **trước** khi lấy dư.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [3]:
for n in [25,26,27,28,52,53]:
    print(n,'//26=',n//26,'%26=',n%26)
    assert (n//26)*26+n%26==n


25 //26= 0 %26= 25
26 //26= 1 %26= 0
27 //26= 1 %26= 1
28 //26= 1 %26= 2
52 //26= 2 %26= 0
53 //26= 2 %26= 1


### 2.3. `divmod(n,26)` trả cả thương và dư

`divmod(a,b)` bằng `(a//b, a%b)` và giúp code rõ nghĩa: `q, r = divmod(n-1,26)`. Phần dư r nằm 0..25 nên ánh xạ thẳng vào `ALPHABET[r]`. Thương q còn lại thể hiện các cột ở phía trái.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [4]:
for n in [1,26,27,28,52,701]:
    q,r=divmod(n-1,26)
    print(f'n={n}: n-1={n-1}, q={q}, r={r}, chữ={ALPHABET[r]}')


n=1: n-1=0, q=0, r=0, chữ=A
n=26: n-1=25, q=0, r=25, chữ=Z
n=27: n-1=26, q=1, r=0, chữ=A
n=28: n-1=27, q=1, r=1, chữ=B
n=52: n-1=51, q=1, r=25, chữ=Z
n=701: n-1=700, q=26, r=24, chữ=Y


### 2.4. `chr()` biến mã Unicode thành ký tự

`ord("A")` trả mã của chữ A; `chr(ord("A")+0)` là A, cộng 25 là Z. Dùng `chr(ord("A") + r)` khi r ∈ 0..25. Đừng dùng số 64/65 như “mẹo” mà không hiểu `ord`/`chr`.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [5]:
print(ord('A'),ord('Z'))
for r in [0,1,25]:print('r',r,'->',chr(ord('A')+r))


65 90
r 0 -> A
r 1 -> B
r 25 -> Z


### 2.5. `list.append()` và `.reverse()`

Các phép chia liên tiếp lấy ký tự **từ phải sang trái**. Vì vậy ta có thể thêm vào một list `letters`, rồi `reverse()` trước khi `"".join(letters)`. `append` không trả về list mới; nó sửa tại chỗ và trả None.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [6]:
letters=[]
letters.append('B');letters.append('A')
print('lúc lấy dư:',letters)
letters.reverse()
print('sau reverse:',letters,'->',''.join(letters))


lúc lấy dư: ['B', 'A']
sau reverse: ['A', 'B'] -> AB


### 2.6. `while n > 0` và phép đo tiến trình

Mỗi bước biến n thành `(n-1)//26`, nhỏ hơn n khi n ≥ 1. Vì vậy vòng lặp hữu hạn. Kiểm tra n > 0 **trước** khi làm `n -= 1` vì đề chỉ cho số dương; nếu n đã 0 thì không có chữ số để chuyển nữa.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [7]:
n=701
while n>0:
    n-=1
    q,r=divmod(n,26)
    print('chữ:',ALPHABET[r],'; thương mới=',q)
    n=q


chữ: Y ; thương mới= 26
chữ: Z ; thương mới= 0


### 2.7. Đọc Big-O theo số ký tự kết quả

Một chữ biểu diễn 26 cột; hai chữ biểu diễn tới 26+26²; số ký tự d tăng xấp xỉ `log_26(n)`, không phải n. Với giới hạn n ≤ 2³¹-1, số ký tự chỉ vài chữ, nhưng khi phân tích tổng quát phải nói d là số ký tự sinh ra.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [8]:
for n in [1,26,27,702,703,18278,18279]:
    d=0;x=n
    while x>0:x=(x-1)//26;d+=1
    print(n,'->',d,'ký tự')


1 -> 1 ký tự
26 -> 1 ký tự
27 -> 2 ký tự
702 -> 2 ký tự
703 -> 3 ký tự
18278 -> 3 ký tự
18279 -> 4 ký tự


## 3. Nền tảng tư duy — từ ví dụ nhỏ đến bài toán

Bảy thí nghiệm ở trên không phải những mẹo rời rạc: chúng chính là các khối xây dựng cho bài #168. Hãy kiểm tra lần lượt: **dữ liệu đang nằm ở đâu, biến nào lưu thông tin đã biết, bước nào khiến ta tiến tới đáp án?**

Khi học thuật toán, đừng chỉ hỏi “code này chạy gì?”; hãy hỏi **tại sao không cần nhớ phần thông tin đã bỏ đi** và vì sao ta không bỏ sót lời giải. Phần sau sẽ chạy một số ví dụ rất nhỏ trước khi ghép thành hàm hoàn chỉnh.


In [9]:
def column_by_hand_168(n):
    """Một bước dịch 1-based về dư 0..25."""
    q,r=divmod(n-1,26)
    return q,chr(ord('A')+r)
for n in [1,26,27,28,701]:
    print(n,'->',column_by_hand_168(n))


1 -> (0, 'A')
26 -> (0, 'Z')
27 -> (1, 'A')
28 -> (1, 'B')
701 -> (26, 'Y')


### Câu hỏi gợi mở trước khi tối ưu

- Vì sao n=26 phải ra Z, không ra A@?
- Tại sao phần dư đầu tiên tương ứng chữ **ngoài cùng bên phải**?
- Làm cách nào để mọi vòng lặp đều sinh được chữ trong khoảng A..Z mà không phải xử lý riêng 26, 52, 78...?

Chưa cần thuộc tên kỹ thuật **Math · bijective base-26 · chia lấy dư**. Trước tiên hãy tưởng tượng bạn chỉ có bút và giấy: bạn đọc input theo thứ tự nào, giữ lại những gì, điều gì buộc bạn dừng? Câu trả lời sẽ trở thành các biến và điều kiện `if`/`while`.


### Quy ước chạy cell và cách tự kiểm tra

- Đọc thứ tự từ trên xuống: các hàm helper trong phần này chỉ dùng để dựng dữ liệu hoặc hiển thị; đáp án submit sẽ không phụ thuộc chúng.
- `print` là quan sát, `return` là kết quả hàm. Dùng `assert actual == expected` để kiểm chứng, không thay thế một phép kiểm bằng việc nhìn thấy chữ PASS.
- Khi thấy nhiều thuật toán, chúng có **tên hàm khác nhau** (`baseline_*`, `solve_*`, `trace_*`) để tránh ghi đè logic. Không thay đổi input trừ khi đề cho phép.
- Đừng chạy hàm TODO trước khi tự hoàn thiện: hàm được để `pass`/`NotImplementedError` có chủ đích.

**Nguyên tắc quan trọng:** ví dụ và test là bằng chứng thử nghiệm; lập luận về bất biến mới giúp giải thích vì sao thuật toán đúng cho mọi input hợp lệ.


### 3.1. Một phép chia đại diện cho đúng một chữ cái

Lấy `n-1`, rồi `divmod(...,26)` tạo `r` từ 0 đến 25. Chữ tương ứng `ALPHABET[r]` thuộc cột phải nhất; thương q là phần chưa chuyển. Viết lại `n = 26*q + (r+1)`, ta thấy `r+1` nằm 1..26 đúng quy tắc Excel. Ví dụ 28 = 26×1 + 2 nên cột cuối là B.

Chạy đoạn code dưới rồi thử tự diễn đạt **trạng thái trước – thao tác – trạng thái sau**. Nếu chỉ nhớ kết quả in ra mà không giải thích được sự chuyển đổi thì nên đọc lại ví dụ.


In [10]:
for n in [26,27,28,52,53]:
    q,r=divmod(n-1,26)
    print(f'{n:3d} = 26*{q} + {r+1:2d} -> chữ cuối {ALPHABET[r]}')


 26 = 26*0 + 26 -> chữ cuối Z
 27 = 26*1 +  1 -> chữ cuối A
 28 = 26*1 +  2 -> chữ cuối B
 52 = 26*1 + 26 -> chữ cuối Z
 53 = 26*2 +  1 -> chữ cuối A


### 3.2. Hai vòng liên tiếp: tính 701 → ZY bằng tay

Với n=701: (701−1)=700, chia 26 được thương 26, dư 24 → Y. Với n=26: (26−1)=25, chia 26 được thương 0, dư 25 → Z. Lấy ngược Y,Z thành ZY. Nếu không đảo chuỗi, bạn sẽ trả YZ, tương ứng cột khác.

Đây là bước nối giữa ví dụ nhỏ và code thuật toán: những biến xuất hiện trong cell tiếp theo sẽ tái xuất hiện trong `solve_*` hoặc `trace_*` ở dưới.


In [11]:
n=701;letters=[]
while n:
    n,r=divmod(n-1,26)
    letters.append(chr(65+r))
    print('thu được chữ',letters[-1],'| còn n=',n)
print('ghép xuôi các chữ thu được:',letters,'->',''.join(reversed(letters)))


thu được chữ Y | còn n= 26
thu được chữ Z | còn n= 0
ghép xuôi các chữ thu được: ['Y', 'Z'] -> ZY


## 4. Cách giải trực giác / Baseline

Cách ngắn, dễ giải thích bằng **đệ quy**: tính chữ cuối sau khi `divmod(n-1,26)`, rồi gọi lại chính hàm với thương q để tạo tiền tố ở bên trái. Nếu q=0 thì không còn tiền tố. Nó đúng và rất sáng sủa, nhưng mỗi lần gọi dùng một frame trên call stack; có thể dùng vòng lặp và list để chủ động hơn.

Ta cố ý viết phiên bản **dễ nghĩ nhất trước**, dù đôi khi nó dùng thêm bộ nhớ hoặc thao tác dư thừa. Lợi ích của baseline: nhìn được định nghĩa bài toán thành code, và có một “đáp án chuẩn” để đối chiếu khi tối ưu.


In [12]:
def baseline_168(n):
    q,r=divmod(n-1,26)
    last=chr(ord('A')+r)
    if q==0:
        return last
    return baseline_168(q)+last


In [13]:
for n in [1,26,27,28,52,53,701,702,703,2147483647]:
    print(f'{n:10d} -> {baseline_168(n)}')
assert baseline_168(2147483647)=='FXSHRXW'


         1 -> A
        26 -> Z
        27 -> AA
        28 -> AB
        52 -> AZ
        53 -> BA
       701 -> ZY
       702 -> ZZ
       703 -> AAA
2147483647 -> FXSHRXW


### 4.1. Đọc baseline và nhận ra chi phí

Đệ quy tạo đúng số khung gọi bằng số chữ d. Thời gian ít nhất O(d); trong Python, mỗi lần ghép string mới tốn độ dài chuỗi ở thời điểm đó, về phân tích chặt có thể tính tổng thành O(d²), dù d tối đa 7 theo giới hạn đề. Ngăn xếp đệ quy dùng O(d). Cách lặp với list tránh sâu đệ quy và rõ thứ tự lấy chữ.

Không nên khẳng định nhanh/chậm chỉ dựa trên việc chạy vài ví dụ. Hãy **đếm bước**, chỉ ra `n` đang đo cái gì và xét input xấu nhất. Sau này khi thử tối ưu, kết quả phải vẫn khớp baseline với cùng một input.


In [14]:
print('Thử các ngưỡng thay số ký tự:')
for n in [25,26,27,51,52,53,701,702,703,18278,18279]:
    print(n,baseline_168(n))


Thử các ngưỡng thay số ký tự:
25 Y
26 Z
27 AA
51 AY
52 AZ
53 BA
701 ZY
702 ZZ
703 AAA
18278 ZZZ
18279 AAAA


### 4.2. Đường vòng / phản ví dụ giúp thấy lỗi tư duy

Lấy `r=n%26`, rồi dùng `chr(64+r)` hoặc `ALPHABET[r-1]` sẽ mắc lỗi ở bội số của 26, vì r bằng 0. Ví dụ `26` phải cho Z, không được cho ký tự `@` hoặc sai vị trí. Trừ 1 **trước khi chia** là cách chữa tận gốc, không phải thêm một nhánh vá lỗi riêng.

Hãy tìm một đầu vào **cực nhỏ** làm giả thuyết sai; dạng test này hiệu quả hơn việc chỉ chạy ngẫu nhiên dữ liệu dài.


In [15]:
def wrong_no_shift_168(n):
    q,r=divmod(n,26)
    return chr(64+r)
for n in [25,26,27,52]:
    print(n,'chữ cuối SAI:',wrong_no_shift_168(n),'| đúng:',column_by_hand_168(n)[1])


25 chữ cuối SAI: Y | đúng: Y
26 chữ cuối SAI: @ | đúng: Z
27 chữ cuối SAI: A | đúng: A
52 chữ cuối SAI: @ | đúng: Z


## 5. Vì sao cần bước cải tiến?

Cả baseline đệ quy và cách vòng lặp đều tận dụng phép chia ở hệ 26 không có chữ số 0. Cải tiến ở đây không phải đổi O(d) thành O(1) — vì bản thân output có d chữ — mà là bỏ call stack đệ quy, chuyển thành vòng lặp có thể trace và kiểm soát bộ nhớ.

Ta không muốn “tối ưu bằng phép màu”. Mục tiêu là **chứng minh có thể bỏ một phần công việc** mà vẫn giữ thông tin đủ để quyết định đúng. Sau khi tìm thấy đại lượng/cấu trúc bắt buộc phải duy trì, ta mới viết thuật toán chính.


### Bước đột phá — Chuyển từ tập chữ 1..26 sang 0..25 bằng `n -= 1`

Mỗi bước: trừ 1, lấy `r=n%26`, sinh `chr(65+r)`, cập nhật `n//=26`. Lặp khi n > 0. Những ký tự sinh ra theo thứ tự ngược vì phép chia tách từ cột cuối lên cột đầu. Đảo danh sách ký tự sau vòng lặp, nối lại thành string.

**Tự kiểm tra:** Nếu làm thêm đúng một bước nữa, cái gì chắc chắn vẫn đúng? Nếu thuật toán dừng ngay bây giờ, vì sao kết quả nó trả là hợp lệ? Hai câu hỏi này chính là hạt nhân của bất biến và điều kiện dừng.


## 6. Thuật toán chính — mổ xẻ từng bước rồi ghép lại

Giải pháp chính dùng biến `chars` để lưu các chữ mới lấy được. Nó tạo mỗi chữ một lần, `append` chi phí hằng số trung bình, đảo list cuối cùng cũng O(d). Cần bảo đảm khi n giảm đến 0, vòng lặp kết thúc và trả về chuỗi không dư ký tự.

Trước tiên xem một phép biến đổi ngắn trong cell nhỏ, sau đó đọc hàm `solve_*` không có `print`. **Không nhảy thẳng đến class Solution**: phần luyện tay và trace phía dưới sẽ làm rõ từng điều kiện.


### 6.1. Đọc từng biến, không chỉ đọc tên hàm

- `chars = []`: danh sách chữ lấy từ cuối lên đầu.
- `while columnNumber > 0`: vẫn còn phần số chưa được biểu diễn.
- `columnNumber -= 1`: dịch cơ số không có số 0 thành 0-based.
- `columnNumber, remainder = divmod(columnNumber,26)`: lấy thương mới và chữ cuối 0..25.
- `chars.append(chr(ord('A') + remainder))`: thêm đúng một chữ vào danh sách.
- `return ''.join(reversed(chars))`: đảo lại thứ tự chữ để được tên cột.

Đặc biệt để ý **thứ tự cập nhật**: đổi thứ tự hai dòng tưởng nhỏ có thể thay đổi ý nghĩa của bất biến. Nếu nhầm, hãy viết trạng thái trước và sau bằng một ví dụ ba phần tử / ba bước.


In [16]:
n=52
n-=1;q,r=divmod(n,26)
print('52: chữ cuối=',chr(65+r),'phần bên trái=',q)
assert chr(65+r)=='Z' and q==1


52: chữ cuối= Z phần bên trái= 1


In [17]:
def solve_168(columnNumber: int) -> str:
    chars=[]
    while columnNumber>0:
        columnNumber-=1
        columnNumber, remainder=divmod(columnNumber,26)
        chars.append(chr(ord('A')+remainder))
    return ''.join(reversed(chars))


### 6.2. Tại sao các nhánh `if`/`while` được viết đúng thứ tự?

`columnNumber -= 1` phải xảy ra **trước** `divmod`, và `chars` nhận chữ của phần dư trước khi tính vòng mới. `reversed(chars)` chỉ đảo thứ tự khi kết thúc, không thay đổi giá trị từng chữ. Vì đề bảo đảm n≥1 nên không cần gắn output tự chọn cho n=0.

Trong một vòng lặp hãy đọc bằng câu tiếng Việt: “**Trước bước này ta đã biết ...; ta kiểm tra ...; nếu ... thì ...; sau bước này ...**”. Có thể thêm `print` để kiểm tra suy luận, nhưng hàm nộp bài phải trả **đúng kiểu** kết quả.


In [18]:
for n in [1,26,27,28,52,53,701,702,703,2147483647]:
    actual=solve_168(n)
    assert actual==baseline_168(n)
    print(n,'->',actual)


1 -> A
26 -> Z
27 -> AA
28 -> AB
52 -> AZ
53 -> BA
701 -> ZY
702 -> ZZ
703 -> AAA
2147483647 -> FXSHRXW


### 6.3. Kiểm tra mấu chốt bằng vài bước tay

Sau k vòng, `chars` chứa chính xác k ký tự cuối của tên cột nhưng theo thứ tự đảo; `n` còn lại là số biểu diễn phần tiền tố. Sự thật này đúng từ `chars=[]`, đúng sau một lần dịch 1 và chia, và khi `n=0` toàn bộ chữ đã được sinh. Vì vậy đảo list là ra đáp án.

Một bất biến đúng phải có ba điều: **đúng lúc bắt đầu; vẫn đúng sau một lần cập nhật; khi dừng thì suy ra đáp án**. Dù chưa biết từ “invariant”, bạn có thể phát biểu tự nhiên như câu trên.


In [19]:
n=2147483647;chars=[]
for k in range(8):
    if n==0:break
    q,r=divmod(n-1,26);chars.append(chr(65+r))
    print('bước',k+1,'n trước=',n,'chữ cuối=',chars[-1],'n sau=',q)
    n=q
assert ''.join(reversed(chars))=='FXSHRXW'


bước 1 n trước= 2147483647 chữ cuối= W n sau= 82595524
bước 2 n trước= 82595524 chữ cuối= X n sau= 3176750
bước 3 n trước= 3176750 chữ cuối= R n sau= 122182
bước 4 n trước= 122182 chữ cuối= H n sau= 4699
bước 5 n trước= 4699 chữ cuối= S n sau= 180
bước 6 n trước= 180 chữ cuối= X n sau= 6
bước 7 n trước= 6 chữ cuối= F n sau= 0


## 7. TRACE từng bước — nhìn thấy trạng thái biến

Ta sẽ dùng chương trình `trace_*` khác với lời giải chính: `solve_*` phục vụ submit và test; `trace_*` chỉ phục vụ việc **quan sát**. Trace phải cho thấy dữ liệu đầu vào, biến *trước khi cập nhật*, điều kiện/nhánh đang xét, biến *sau khi cập nhật*, và điều kiện dừng.

Trace chuẩn với 701 gồm **hai lần chia**. Cột n trước là số cột còn phải giải mã; `n-1` là số vừa dịch; q,r là thương/dư; ký tự mới là **chữ bên phải** của kết quả. Sau bước cuối, thương 0 thì dừng.


In [20]:
def trace_168(n):
    chars=[];step=0
    print('step | n_trước | n-1 | q | r | chữ_mới | chars_đảo')
    while n>0:
        before=n
        n,r=divmod(n-1,26)
        chars.append(chr(65+r));step+=1
        print(f'{step:4d} | {before:7d} | {before-1:7d} | {n:3d} | {r:2d} | {chars[-1]:7s} | {chars}')
    out=''.join(reversed(chars))
    print('DỪNG ->',out)
    return out
assert trace_168(701)=='ZY'


step | n_trước | n-1 | q | r | chữ_mới | chars_đảo
   1 |     701 |     700 |  26 | 24 | Y       | ['Y']
   2 |      26 |      25 |   0 | 25 | Z       | ['Y', 'Z']
DỪNG -> ZY


### 7.1. Trace dạng bảng — diễn giải cột và thứ tự cập nhật

Ví dụ 701: hàng 1 `n=701`, `n-1=700`, `q=26`,`r=24`, chữ `Y`; hàng 2 `n=26`, `n-1=25`, `q=0`, `r=25`, chữ `Z`. Danh sách tạm là `[Y,Z]`; đảo lại thành `ZY`. Cột q ở hàng này trở thành n trước của hàng sau.

Khi đọc bảng, đừng coi một hàng là một đáp án; một hàng biểu diễn **một phép chuyển trạng thái**. Hãy tự che cột cuối, dự đoán hành động, rồi đối chiếu với print phía dưới.


In [21]:
for n in [26,27,52,53]:
    print('TRACE n=',n)
    assert trace_168(n)==solve_168(n)


TRACE n= 26
step | n_trước | n-1 | q | r | chữ_mới | chars_đảo
   1 |      26 |      25 |   0 | 25 | Z       | ['Z']
DỪNG -> Z
TRACE n= 27
step | n_trước | n-1 | q | r | chữ_mới | chars_đảo
   1 |      27 |      26 |   1 |  0 | A       | ['A']
   2 |       1 |       0 |   0 |  0 | A       | ['A', 'A']
DỪNG -> AA
TRACE n= 52
step | n_trước | n-1 | q | r | chữ_mới | chars_đảo
   1 |      52 |      51 |   1 | 25 | Z       | ['Z']
   2 |       1 |       0 |   0 |  0 | A       | ['Z', 'A']
DỪNG -> AZ
TRACE n= 53
step | n_trước | n-1 | q | r | chữ_mới | chars_đảo
   1 |      53 |      52 |   2 |  0 | A       | ['A']
   2 |       2 |       1 |   0 |  1 | B       | ['A', 'B']
DỪNG -> BA


### 7.2. Một nhánh đặc biệt mà cách học thuộc code dễ bỏ lỡ

Tại `n=52`, phép chia nguyên 52//26=2, dư 0 nếu không dịch. Nhưng tiêu đề đúng là AZ, không phải một chuỗi có dấu @. Nhìn n=52 trừ 1 thành 51: 51//26=1, 51%26=25 → Z, rồi n=1 → A.

Câu hỏi sau khi chạy: **Nếu bỏ điều kiện xử lý nhánh này, input nhỏ nhất nào trả sai?**


In [22]:
assert trace_168(52)=='AZ'
assert trace_168(702)=='ZZ'
assert trace_168(703)=='AAA'


step | n_trước | n-1 | q | r | chữ_mới | chars_đảo
   1 |      52 |      51 |   1 | 25 | Z       | ['Z']
   2 |       1 |       0 |   0 |  0 | A       | ['Z', 'A']
DỪNG -> AZ
step | n_trước | n-1 | q | r | chữ_mới | chars_đảo
   1 |     702 |     701 |  26 | 25 | Z       | ['Z']
   2 |      26 |      25 |   0 | 25 | Z       | ['Z', 'Z']
DỪNG -> ZZ
step | n_trước | n-1 | q | r | chữ_mới | chars_đảo
   1 |     703 |     702 |  27 |  0 | A       | ['A']
   2 |      27 |      26 |   1 |  0 | A       | ['A', 'A']
   3 |       1 |       0 |   0 |  0 | A       | ['A', 'A', 'A']
DỪNG -> AAA


### 7.3. Trường hợp biên — không chỉ là dữ liệu “đẹp”

n=1 là nhỏ nhất và sinh đúng A một lần. N = 26 là Z; n=27 là AA; n=703 là AAA. Không cần xử lý riêng các ngưỡng bởi công thức dịch 1 hoạt động thống nhất. Input n=0 nằm ngoài ràng buộc của bài gốc.

Trường hợp biên không phải ngoại lệ vô nghĩa: nó bộc lộ một giả định ẩn trong điều kiện lặp hoặc cách tính chỉ số. Chỉ thêm xử lý khi ràng buộc thực sự cho phép trường hợp đó.


In [23]:
for n,expected in [(1,'A'),(26,'Z'),(27,'AA'),(702,'ZZ'),(703,'AAA')]:
    actual=solve_168(n)
    print(n,actual,'PASS' if actual==expected else 'FAIL')
    assert actual==expected


1 A PASS
26 Z PASS
27 AA PASS
702 ZZ PASS
703 AAA PASS


### 7.4. Đối chiếu baseline và tối ưu trên một bộ input

`baseline_168` triển khai đệ quy, còn `solve_168` dùng vòng lặp. Cả hai cùng dựa trên phép chia đã dịch `n-1`, nhưng quản lý trạng thái khác nhau. Đây là cơ hội kiểm chứng rằng hai kiểu viết đều biểu diễn cùng một quan hệ toán học.

Nếu hai lời giải cho kết quả khác nhau, chưa kết luận ngay “baseline sai”; hãy in trạng thái tại input phản ví dụ, đối chiếu lại định nghĩa, rồi sửa mệnh đề bất biến.


In [24]:
for n in [1,2,25,26,27,52,53,700,701,702,703,18278,18279,2147483647]:
    assert baseline_168(n)==solve_168(n)
print('Hai cách giải đối chiếu trên ngưỡng: PASS')


Hai cách giải đối chiếu trên ngưỡng: PASS


## 8. Vì sao thuật toán đúng? — chứng minh ngắn nhưng đủ ý

Gọi n>0. Ta có `n-1=26q+r` với `0≤r≤25`, suy ra `n=26q+(r+1)` và `r+1` là chữ phải nhất của tên cột. Vì vậy mỗi bước trích đúng chữ cuối, không cần đoán. Thương q biểu diễn phần tiền tố và nhỏ hơn n, nên thuật toán chắc chắn dừng. **Bất biến:** sau k lần, phần đã thêm vào `chars` là k chữ cuối theo thứ tự ngược; n chứa tiền tố chưa xử lý. Khi n=0 không còn tiền tố; đảo `chars` tạo tên đầy đủ và duy nhất.

**Phân biệt:** `assert` minh họa “đã chạy đúng các ca này”, không chứng minh hết vô hạn input. Chứng minh dựa vào sự thật được giữ qua mọi lần lặp và điều kiện dừng; đó là kỹ năng sẽ dùng lại ở nhiều bài LeetCode khác.


## 9. Code nộp LeetCode (Python 3)

Đây là phiên bản **không in debug**, trả về đúng dữ liệu cần thiết, giữ đúng tên phương thức `convertToTitle` và tên tham số của đề. Khi submit, copy **chỉ** `class Solution` (và import nếu có dùng); với bài dùng `ListNode`, LeetCode đã cung cấp định nghĩa kiểu node.

Sau cell này, notebook sẽ gọi chính `Solution().convertToTitle(...)` bằng `assert` để kiểm tra cách đóng gói, tránh trường hợp hàm minh họa đúng nhưng chữ ký nộp sai.


In [25]:
class Solution:
    def convertToTitle(self, columnNumber: int) -> str:
        chars=[]
        while columnNumber>0:
            columnNumber,remainder=divmod(columnNumber-1,26)
            chars.append(chr(ord('A')+remainder))
        return ''.join(reversed(chars))


## 10. Kiểm thử: mẫu đề, biên, phản ví dụ, kiểm thử đối chiếu

Bắt buộc kiểm tra 1, 26, 27, 52, 53, 701, 702, 703, và cận trên. Các ngưỡng này giúp phát hiện sai lầm `%26=0` và quên đảo chuỗi. Phép so là string chính xác, phân biệt AZ và ZA.

Với một ca test, viết rõ **input → expected → actual**. Ưu tiên các ca làm lộ bug thay vì lặp lại hai ví dụ thông thường. Test với input gốc theo đúng kiểu LeetCode, không chỉ kiểm tra các biến đã được biến đổi trong demo.


In [26]:
cases={1:'A',2:'B',26:'Z',27:'AA',28:'AB',52:'AZ',53:'BA',701:'ZY',702:'ZZ',703:'AAA',2147483647:'FXSHRXW'}
for n,expected in cases.items():
    assert Solution().convertToTitle(n)==expected,(n,expected)
print('Kiểm thử ví dụ và ngưỡng:',len(cases),'PASS')


Kiểm thử ví dụ và ngưỡng: 11 PASS


### 10.1. Test hệ thống trên một miền nhỏ

Chạy tất cả n=1..10.000 và so với cách đệ quy. Đây là kiểm thử vét cạn **trên đoạn đó**, không phải mọi số nguyên cho phép. Nó sẽ chạm nhiều bội 26 và nhiều lần tăng độ dài chữ.

Với các input nhỏ, có thể duyệt toàn bộ cấu hình thỏa ràng buộc để tìm phản ví dụ. Nếu không thể duyệt tất cả do số lượng quá lớn, ta kiểm tra toàn bộ một miền con và nêu rõ phạm vi đó, **không gọi nó là chứng minh tổng quát**.


In [27]:
for n in range(1,10001):
    assert solve_168(n)==baseline_168(n)
print('Vét cạn 1..10.000: PASS')


Vét cạn 1..10.000: PASS


### 10.2. Kiểm thử ngẫu nhiên **có seed** và so sánh với oracle

Lấy 500 số ngẫu nhiên trên toàn miền dương 32-bit và so baseline. Seed 168 giúp chạy lại đúng bộ. Bổ sung test chuyển đổi hai chiều bằng cách tự viết hàm đọc chữ theo từng vị trí (oracle).

`random.Random(seed)` khiến bộ test có thể chạy lại giống hệt. **Oracle** là một cách kiểm tra độc lập, thường chậm hơn nhưng dễ kiểm chứng. Không dùng hai code giống hệt nhau để đối chiếu — nếu cùng mắc lỗi, test sẽ cho cảm giác an toàn giả.


In [28]:
def decode_168(text):
    x=0
    for ch in text:
        x=x*26 + (ord(ch)-ord('A')+1)
    return x
rng=random.Random(168)
for _ in range(500):
    n=rng.randint(1,2**31-1)
    s=solve_168(n)
    assert s==baseline_168(n)
    assert decode_168(s)==n
print('Random 500 + round-trip: PASS')


Random 500 + round-trip: PASS


### 10.3. Tính chất khác của đáp án và dữ liệu đầu vào

Chuyển đổi có tính đơn ánh trên các số nguyên dương: nếu mã hóa rồi giải mã thì trở lại số ban đầu. Độ dài output chỉ tăng tại các ngưỡng 27, 703, 18279..., và mỗi chữ luôn nằm A..Z. Hàm không sửa số nguyên input vì `int` là immutable.

Không chỉ hỏi “trả đúng số chưa?”; đôi khi đề còn yêu cầu **không sửa cấu trúc** hoặc có ràng buộc độ dài/bit. Kiểm thử thuộc tính giúp phát hiện lỗi chỉ xuất hiện ở nhóm input đặc biệt.


In [29]:
for n in [1,26,27,702,703,18278,18279,2147483647]:
    s=solve_168(n)
    assert s.isupper() and s.isalpha()
    assert 1<=len(s)<=7
    assert decode_168(s)==n
print('Thuộc tính round-trip và ký tự hợp lệ: PASS')


Thuộc tính round-trip và ký tự hợp lệ: PASS


## 11. Phân tích Time / Space Complexity

Gọi **d** là số ký tự trong tên cột, d≈log_26(n). Mỗi vòng `n=(n-1)//26` nên chạy d lần. **Vòng lặp + list:** Time O(d), Space O(d) để lưu chữ kết quả; phụ trợ không tính output không đáng kể ngoài danh sách ký tự. Nếu coi nối string output bắt buộc, cần ít nhất Ω(d) thời gian/đầu ra. **Baseline đệ quy:** O(d) khung stack; do nối string mỗi tầng, trong mô hình sao chép string có thể lên O(d²) thời gian. Với giới hạn 32 bit, d≤7, nhưng phân tích tổng quát theo d giúp hiểu thuật toán.

Để tránh nói Big-O máy móc: chỉ rõ biến kích thước input, số vòng chạy, số bộ nhớ phụ thực sự cấp phát. Nếu đầu ra có độ dài bắt buộc, phân biệt **bộ nhớ phụ** với **bộ nhớ dành cho kết quả**; nếu 32 bit là hằng số, đừng vô tình coi đó là biến tăng vô hạn.


## 12. Sai lầm người mới hay mắc — code sai, vì sao sai, cách sửa

Các lỗi quan trọng: dùng `%26` trực tiếp gây dư 0; lấy đúng chữ nhưng quên đảo thứ tự; nhầm cột Excel 1-based với index Python 0-based; viết `while n>=0` gây thêm chữ sai khi n=0; trả list thay vì string.

Không lấy việc có kết quả đúng trên ví dụ đầu làm lý do bỏ qua các lỗi sau. Với từng lỗi, chỉ ra một input phản chứng; chạy đoạn minh họa riêng, sau đó đối chiếu với `solve_*` đã được test.


In [30]:
for n in [26,52,702]:
    print(n,'dư thường=',n%26,'dư sau dịch=',(n-1)%26)
    assert (n-1)%26==25


26 dư thường= 0 dư sau dịch= 25
52 dư thường= 0 dư sau dịch= 25
702 dư thường= 0 dư sau dịch= 25


### 12.1. Lỗi thứ nhất: quên trừ 1 trước khi chia

Dùng n%26 ở bội 26 trả dư 0. Nếu ánh xạ 0→A thì sẽ ghi sai, vì cột 26 phải là Z. Không vá bằng cách “if dư 0: ghi Z và giảm thương 1” khi đã có cách biến đổi thống nhất: `divmod(n-1,26)`.

**Cách sửa:** quay lại định nghĩa và điều kiện dừng, không chỉ thêm một `if` để vá riêng input mẫu.


In [31]:
def bad_168(n):
    letters=[]
    while n:
        n,r=divmod(n,26)
        letters.append(chr(65+r))
    return ''.join(reversed(letters))
# Không gọi bad_168(26): r=0 rồi n=1, cuối cùng sinh 'BA' sai.
print('Phản ví dụ an toàn 28:',bad_168(28),'vs',solve_168(28))
assert bad_168(28)!=solve_168(28)


Phản ví dụ an toàn 28: BC vs AB


### 12.2. Lỗi thứ hai: đảo thứ tự quá sớm hoặc không đảo

Phần dư tương ứng chữ **bên phải**, nhưng list tích lũy từ phải qua trái. Nếu `''.join(chars)` ngay sẽ trả YZ cho 701, thay vì ZY. Giữ `append`, rồi `reversed(chars)` đúng một lần tại return.

**Cách sửa:** lập bảng một hoặc hai bước đầu để biết một phép tính dùng trạng thái *trước* hay *sau* cập nhật. Khi đã thấy lỗi, ghi lại một regression test để bug không quay lại.


In [32]:
n=701;chars=[]
while n:
    n,r=divmod(n-1,26)
    chars.append(chr(65+r))
print('SAI nếu nối ngay:', ''.join(chars))
print('ĐÚNG nếu đảo:', ''.join(reversed(chars)))
assert ''.join(chars)=='YZ' and ''.join(reversed(chars))=='ZY'


SAI nếu nối ngay: YZ
ĐÚNG nếu đảo: ZY


## 13. Nhận diện pattern và tự suy ra kỹ thuật ở bài khác

Dạng bài **đổi biểu diễn giữa số nguyên và hệ vị trí chữ**: nghĩ đến chia lấy thương/phần dư để trích chữ từ phải; nếu ký hiệu bắt đầu từ 1, phải dịch 1 trước khi chia. Dạng ngược #171 dùng phép **nhân 26 rồi cộng chữ số**, sẽ học ngay trong bộ này. Đừng nhầm với hệ nhị phân có chữ số 0 hợp lệ.

Một kỹ thuật không gắn với duy nhất bài #168: điều cần ghi nhớ là **dấu hiệu trong đề**, **trạng thái tối thiểu cần giữ**, **quy tắc cập nhật**, **khi nào dừng**, và **mức độ phức tạp có thể đạt được**. Thử tự nói thành 4–5 câu thay vì ghi nhớ code.


## 14. Biến thể — thay đổi yêu cầu để kiểm tra hiểu thật

Học phép đảo ngược: từ tên cột Excel về số, kiểm tra round-trip. Sau đó mở rộng thành “tiêu đề cột liền sau” để thấy hai hàm giải mã/mã hóa kết hợp thành một công cụ.

Nếu hiểu thuật toán, bạn có thể thay đổi mục tiêu mà vẫn suy ra cách làm. Phần này chỉ giới thiệu biến thể **có ý nghĩa và nằm ngoài yêu cầu gốc**; code submit phía trên vẫn bám đúng đề LeetCode.


In [33]:
for title in ['A','Z','AA','AZ','ZY','FXSHRXW']:
    n=decode_168(title)
    print(title,'→',n,'→',solve_168(n))


A → 1 → A
Z → 26 → Z
AA → 27 → AA
AZ → 52 → AZ
ZY → 701 → ZY
FXSHRXW → 2147483647 → FXSHRXW


### 14.1. Biến thể thứ nhất: đổi ngược từ tên cột sang số

Đọc trái→phải, `value = value*26 + digit`, trong đó digit = `ord(letter)-ord('A')+1`. Đây chính là bài #171; nó cho phép xác nhận phép mã hóa của #168 không đánh mất thông tin.

Hãy xác định đầu ra mới có cần nhớ thêm trạng thái không. Nếu có, giải thích vì sao; nếu không, chỉ ra biến đang đủ dùng.


In [34]:
def title_to_number_168(s):
    number=0
    for ch in s:
        number=number*26 + ord(ch)-ord('A')+1
    return number
for n in [1,26,27,701,702,703]:
    assert title_to_number_168(solve_168(n))==n
print('Biến thể giải mã: PASS')


Biến thể giải mã: PASS


### 14.2. Biến thể thứ hai: tiêu đề cột tiếp theo trong Excel

Với tên cột hiện tại, trước tiên giải thành số n, cộng 1, rồi đổi số trở lại tên cột. Đây là cách dễ hiểu, dù không trực tiếp thao tác trên chuỗi ký tự. Ví dụ Z→AA, AZ→BA, ZZ→AAA.

Khi thay đổi ràng buộc đầu vào, các chứng minh cho bài gốc có thể **không còn đúng**; cần sửa chứng minh trước khi sửa code.


In [35]:
def next_column_168(s):
    return solve_168(title_to_number_168(s)+1)
for a,b in [('A','B'),('Z','AA'),('AZ','BA'),('ZZ','AAA')]:
    print(a,'→',next_column_168(a))
    assert next_column_168(a)==b


A → B
Z → AA
AZ → BA
ZZ → AAA


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

**Bài 1 — dự đoán output:** Dự đoán `divmod(26-1,26)`, `chr(ord("A")+25)` và `"".join(reversed(["Y","Z"]))`. **Đừng chạy ngay:** ghi kết quả dự đoán, rồi đối chiếu với code nền tảng.

**Bài 2 — tính tay / trace:** Tự lập bảng n, n−1, q, r, chữ cho 52 và 703. Ghi ít nhất ba cột *trạng thái trước / thao tác / trạng thái sau*.

**Bài 3 — TODO viết lời giải chính:** hoàn thiện hàm dưới đây mà **không nhìn phần đáp án**.

**Bài 4 — biến thể:** thay đổi kiểu output hoặc điều kiện nhưng dùng lại mấu chốt thuật toán.

**Bài 5 — tự tạo kiểm thử:** nghĩ ít nhất 5 ca phân biệt được code đúng/sai, có ca biên và ca phản chứng.

**Bài 6 — mini challenge:** tự tổng quát hóa kỹ thuật, thử phân tích độ phức tạp và đối chiếu với baseline.

> Các hàm `exercise_*` bên dưới cố ý chưa có lời giải. Chúng **không được gọi tự động** khi chạy notebook. Đáp án chỉ xuất hiện ở mục 17.


### Bài 3 — TODO tự viết lời giải #168

Viết `exercise_title_168(n)` bằng vòng while và list ký tự, không dùng hàm `solve_168` hay `baseline_168`. Điều kiện: n nguyên dương, trả string, không thêm ký tự thừa.

Gợi ý tối thiểu: trước khi viết, hãy ghi bất biến bằng một câu tiếng Việt; hàm bắt buộc trả đúng loại dữ liệu yêu cầu. Sau khi tự viết, bỏ `pass` và test thủ công với ví dụ và ca biên ở mục 10.


In [36]:
def exercise_title_168(n):
    # TODO 1: tạo danh sách letters
    # TODO 2: lặp khi n > 0, trừ 1, divmod(n,26)
    # TODO 3: thêm chữ A..Z
    # TODO 4: đảo và nối thành string
    pass


### Bài 4 — TODO biến thể

Viết `exercise_decode_168(s)` nhận chuỗi A..Z và trả lại số cột. Gợi ý dùng bất biến `number` là giá trị của đoạn tiền tố đã đọc; hãy thử A, AB và ZY.

Bạn được thay kiểu đầu ra nhưng không được thay định nghĩa bài gốc. Để hàm có chữ ký riêng và **không gọi tự động**; tạo dữ liệu thử sau khi đã có code.


In [37]:
def exercise_decode_168(s):
    # TODO: for ch in s, cập nhật number = number*26 + digit
    pass


### Bài 5 — TODO tự tạo test có chủ đích

Tạo bảng ít nhất 5 ranh giới: `1`, `26`, `27`, `52`, `53`, `702`, `703`; so với expected string. Với bài này, bội của 26 là nhóm ca dễ bắt lỗi nhất.

Thử đặt câu hỏi: một chương trình sai phổ biến có thể vẫn vượt qua ca nào? Cần thêm ca nào để khiến nó lộ sai? Viết kiểm thử `assert` trong hàm độc lập, sau đó chỉ gọi khi đã hoàn thiện hàm TODO 3, 4.


In [38]:
def exercise_tests_168():
    # TODO: viết >=5 assert, bao gồm n=26,52,702
    pass


### Bài 6 — mini challenge chuyển giao kỹ thuật

Tổng quát từ bộ 26 chữ sang **bộ chữ bất kỳ có k ký hiệu** (k≥2, tất cả ký hiệu là một ký tự khác nhau), vẫn không có chữ số 0. Ví dụ hệ A,B,C (k=3): 1→A,3→C,4→AA. Hãy viết hàm `exercise_bijective_168(n, alphabet)`.

Đây là bài nâng cao: bạn không cần giải nhanh. Hãy trình bày ý tưởng bằng tay, viết code sạch, tự mô phỏng input nhỏ rồi mới so sánh với đáp án tham khảo cuối notebook.


In [39]:
def exercise_bijective_168(n, alphabet):
    # TODO: thay 26 bằng k=len(alphabet), tra bảng alphabet[r]
    pass


## 16. Gợi ý ba mức — chỉ mở sau khi tự thử

**Bài 1:** M1: phần dư 0..25. M2: 25//26=0 và 25%26=25. M3: `(0,25), Z, ZY`.

**Bài 2:** M1: từng vòng giảm n. M2: r là chữ cuối. M3: 52→Z, n=1→A; 703→A, n=27→A, n=1→A.

**Bài 3:** M1: trừ 1. M2: `n,r=divmod(n-1,26)`. M3: `''.join(reversed(chars))`.

**Bài 4:** M1: chữ 1..26. M2: `ord(ch)-ord('A')+1`. M3: `value=value*26+digit`.

**Bài 5:** M1: bội 26. M2: thử 26,52,702. M3: expected `Z,AZ,ZZ`.

**Bài 6:** M1: 26 không phải số đặc biệt của thuật toán. M2: thay 26 bằng k. M3: dùng `alphabet[r]` với r0-based.

**Cách tự đánh giá:** Nếu mức 1 đã đủ để bạn làm được, hãy che mức 2–3. Nếu phải dùng mức 3, thử giải lại từ đầu sau khi đóng notebook. Học để có thể **tự suy ra**, không phải chép một đoạn code đã biết.


## 17. Đáp án tham khảo — đặt cuối để tránh lộ trước khi làm TODO

Phần này dùng **tên hàm `answer_*` khác với `exercise_*`** để notebook vẫn chạy an toàn. Đáp án không cần giống hệt từng ký tự code của bạn, nhưng phải giữ đúng điều kiện và chứng minh cùng độ phức tạp.

Đáp án giữ đúng cơ số 26 kiểu 1..26 và lấy chữ từ phải. Bài 4 giải mã theo quy tắc cộng dồn; bài 6 cho phép đổi alphabet.


In [40]:
def answer_title_168(n):
    chars=[]
    while n>0:
        n,r=divmod(n-1,26)
        chars.append(chr(65+r))
    return ''.join(reversed(chars))

assert answer_title_168(2147483647)=='FXSHRXW'
print('Đáp án bài 3: PASS')


Đáp án bài 3: PASS


### Đáp án bài 4 — suy ra từ biến thể

Sau khi đọc một tiền tố có giá trị x, nối thêm chữ d nghĩa là `new = 26*x + d`. Tại đầu x=0; đọc hết s thì x chính là số cột. Không được dùng index 0-based làm digit cuối cùng.

Hãy so sánh với phần bạn tự viết: code nào dễ nhìn ra bất biến hơn? Cả hai đúng vẫn có thể khác nhau về chi phí bộ nhớ hoặc sự rõ ràng.


In [41]:
def answer_decode_168(s):
    x=0
    for ch in s:
        x=x*26+(ord(ch)-ord('A')+1)
    return x
for x in [1,26,27,52,53,701,702,703,2147483647]:
    assert answer_decode_168(answer_title_168(x))==x
print('Đáp án bài 4: PASS')


Đáp án bài 4: PASS


### Đáp án bài 6, và cách kiểm tra bài 5

Ký hiệu `[A,B,C]` có k=3. Không có ký hiệu số 0, vì thế vẫn trừ 1 trước khi divmod; thay `chr(...)` bằng `alphabet[r]`. Thử 3→C, 4→AA, 12→CC, 13→AAA.

Mỗi test có một mục đích. Khi tự thiết kế test, ghi chú ca nào kiểm điều kiện dừng, ca nào kiểm cập nhật, ca nào kiểm dữ liệu hợp lệ khác thường.


In [42]:
def answer_bijective_168(n,alphabet):
    k=len(alphabet)
    assert k>=2 and len(set(alphabet))==k
    out=[]
    while n:
        n,r=divmod(n-1,k)
        out.append(alphabet[r])
    return ''.join(reversed(out))
for n,s in [(1,'A'),(3,'C'),(4,'AA'),(12,'CC'),(13,'AAA')]:
    assert answer_bijective_168(n,'ABC')==s,(n,s)
print('Đáp án bài 6: PASS')


Đáp án bài 6: PASS


### Đáp án / lời giải thích bằng lời cho cả 6 bài

**Bài 1:** `divmod(25,26)=(0,25)`, `chr(65+25)="Z"`, đảo `["Y","Z"]` thành `"ZY"`.

**Bài 2:** 52→Z rồi A thành AZ; 703→A, tiếp A rồi A thành AAA. Ghi q sau mỗi vòng làm n của lần sau.

**Bài 3:** Chọn trạng thái tối thiểu như `solve_*`; đọc lại cell đáp án ngay trên và giải thích từng phép gán.

**Bài 4:** Đọc từng chữ từ trái sang phải, nhân tổng cũ với 26 và cộng giá trị 1..26 của chữ.

**Bài 5:** Dùng ca phân biệt các ranh giới 26/27, 52/53, 702/703 và cận trên.

**Bài 6:** Hệ bijective base-k chỉ đổi 26 thành k và bảng chữ, giữ nguyên bước trừ 1.

> Nếu chưa thể tự diễn giải được các dòng `return`, hãy quay lại bảng trace và lặp lại bằng một input tự chọn.


## 18. Tổng kết — template suy nghĩ + checklist nộp bài

### Template nhớ nhanh cho #168

1. **Định nghĩa:** Nhận số nguyên cột n≥1, trả tiêu đề Excel chuỗi A..Z, AA...
2. **Trạng thái:** n chưa giải mã và list chars ghi các chữ cuối theo thứ tự ngược.
3. **Quy tắc cập nhật:** n,r=divmod(n−1,26), thêm chr(ord(A)+r).
4. **Bất biến:** chars chứa các chữ cuối đúng, n còn lại mã hóa tiền tố.
5. **Điều kiện dừng:** Khi n=0, đảo chars và nối thành string.
6. **Độ phức tạp:** O(d) thời gian và O(d) bộ nhớ chứa output; d là số chữ ≈ log_26(n).

### Checklist tự đánh giá
- [ ] Tôi phát biểu được đề, output và ràng buộc mà không nhìn code.
- [ ] Tôi hiểu cú pháp mới và tự chạy được ít nhất hai thí nghiệm nhỏ.
- [ ] Tôi trình bày được baseline, điểm yếu và lý do cải tiến.
- [ ] Tôi tự trace được 4 bước, kể cả một nhánh đặc biệt.
- [ ] Tôi phân biệt được chứng minh thuật toán với kiểm thử.
- [ ] Tôi biết khai báo `class Solution` đúng tên hàm, đúng kiểu trả về.
- [ ] Tôi đã làm 6 bài tự luyện **trước khi** xem đáp án.
- [ ] Tôi kiểm tra lại edge cases, time/space, không để print debug khi submit.

**Câu hỏi cuối:** Nếu thay input và biến thể, tôi có tự viết lại thuật toán mà không nhìn notebook được không? Nếu có, bạn đã hiểu kỹ thuật, không chỉ thuộc code.
